# UPPP 135 - Week 9: Geodemographics

<a target="_blank" href="https://colab.research.google.com/github/knaaptime/uppp135-winter26-assn/blob/main/week9/geodemographics.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Data

In [ ]:
# !pip install geosnap
import geosnap as gsp
import pandas as pd
from IPython.display import Image

datasets = gsp.DataStore()

la = gsp.io.get_acs(datasets, msa_fips="31080", years=[2012, 2017, 2021], level="tract")

la20 = la[la.year == 2021]

In [ ]:
gsp.visualize.plot_timeseries(la, column="median_home_value", nrows=1, ncols=3, figsize=(12, 8))

## Animating Change over Time

In [ ]:
gsp.visualize.animate_timeseries(la, column="median_home_value", figsize=(8, 10), filename='la_homeval_change.gif')

In [ ]:
Image('la_homeval_change.gif')

In [ ]:
gsp.visualize.animate_timeseries(la, column="median_household_income", figsize=(8, 10), filename='la_inc_change.gif')

In [ ]:
Image('la_inc_change.gif')

In [ ]:
gsp.visualize.animate_timeseries(la, column="p_nonhisp_white_persons", figsize=(8, 10), filename='la_white_change.gif')

In [ ]:
Image('la_white_change.gif')

In [ ]:
gsp.visualize.animate_timeseries(la, column="p_nonhisp_black_persons", figsize=(8, 10), filename='la_black_change.gif')

In [ ]:
Image('la_black_change.gif')

In [ ]:
gsp.visualize.animate_timeseries(la, column="p_hispanic_persons", figsize=(8, 10), filename='la_hisp_change.gif')

In [ ]:
Image('la_hisp_change.gif')

In [ ]:
gsp.visualize.animate_timeseries(la, column="p_asian_persons", figsize=(8, 10), filename='la_asian_change.gif')

In [ ]:
Image('la_asian_change.gif')

## Exercise

How has the LA region changed in neighborhoods with a share of population >=60?

How has the LA region changed in neighborhoods with a share of population <=19?

## Creating a typology

### Clustering *one* variable

remember the fisher-jenks classifier is a one-dimensional classifier

In [ ]:
la20[["median_household_income", "geometry"]].explore(
    "median_household_income",
    scheme="fisherjenks",
    k=5,
    style_kwds={"weight": 0.2},
    tiles="cartodb darkmatter",
)

In [ ]:
inc_clusters = gsp.analyze.cluster(
    la20, method="kmeans", columns=["median_household_income"], n_clusters=5
)

inc_clusters[["kmeans", "geometry"]].explore(
    "kmeans",
    style_kwds={"weight": 0.2},
    tiles="cartodb darkmatter",
)

In [ ]:
inc_clusters[["kmeans", "geometry"]].explore(
    "kmeans",
    categorical=True,
    style_kwds={"weight": 0.2},
    tiles="cartodb darkmatter",
    cmap='tab20b'
)

### Clustering multiple variables

In [ ]:
cols = [
    "median_home_value",
    "median_household_income",
    "p_edu_college_greater",
    "p_persons_under_18",
    "p_married",
    "p_persons_over_60",
]

In [ ]:
k5_clusters = gsp.analyze.cluster(la20, method="kmeans", columns=cols, n_clusters=5)
k5_clusters[["kmeans", "geometry"]].explore(
    "kmeans",
    categorical=True,
    style_kwds={"weight": 0.2},
    tiles="cartodb darkmatter",
    cmap='tab20b'
)

### Describing the types

In [ ]:
gsp.visualize.plot_violins_by_cluster(
    k5_clusters, columns=cols, cluster_col="kmeans", nrows=2, ncols=3, figsize=(12, 6)
)

### A different solution

In [ ]:
ward_clusters = gsp.analyze.cluster(la20, method="ward", columns=cols, n_clusters=5)
ward_clusters[["ward", "geometry"]].explore(
    "ward",
    categorical=True,
    style_kwds={"weight": 0.2},
    tiles="cartodb darkmatter",
    cmap="Accent_r",
)

### A different solution's types

In [ ]:
gsp.visualize.plot_violins_by_cluster(
    ward_clusters, columns=cols, cluster_col="ward", nrows=2, ncols=3, figsize=(12, 6)
)

In [ ]:
best_k, bk_table = gsp.analyze.find_k(la, columns=cols, method='ward', return_table=True)

In [ ]:
best_k

In [ ]:
bk_table

In [ ]:
bk_table.silhouette_score.plot()

In [ ]:
best_ward_clusters = gsp.analyze.cluster(la20, method="ward", columns=cols, n_clusters=6)
best_ward_clusters[["ward", "geometry"]].explore(
    "ward",
    categorical=True,
    style_kwds={"weight": 0.2},
    tiles="cartodb darkmatter",
    cmap="Accent",
)

In [ ]:
gsp.visualize.plot_violins_by_cluster(
    best_ward_clusters, columns=cols, cluster_col="ward", nrows=2, ncols=3, figsize=(12, 6)
)

### Exercise:

what if we're interested in multidimensional segregation, which we define as separation between people making different incomes and belonging to different racial/ethnic groups. How would we create a neighborhood typology using the `kmeans` method? Assume our columns are 

`['median_home_value', 'median_household_income', 'p_nonhisp_white_persons', 'p_nonhisp_black_persons', 'p_hispanic_persons', 'p_asian_persons']`

and you are interested in finding `5` neighborhood types. Create a typology using the methods similar to what we did above, then use the `explore` method to plot a map of the resulting clusters.

### Change over Time

In [ ]:
ward_clusters = gsp.analyze.cluster(la, method="ward", columns=cols, n_clusters=8)
ward_clusters[["ward", "geometry"]].explore(
    "ward",
    categorical=True,
    style_kwds={"weight": 0.2},
    tooltip=False,
    tiles="cartodb darkmatter",
    cmap="tab20b",
)

gsp.visualize.animate_timeseries(
    ward_clusters, column="ward", categorical=True, cmap='tab20b', figsize=(8, 10), filename="la_ward_change.gif"
)

In [ ]:
Image('./la_ward_change.gif')

## Exercise: how would we visualize the typology created above?


# Regionalization


### Ward clusters with a spatial constraint

In [ ]:

ward_spatial_clusters = gsp.analyze.regionalize(
    la20, method="ward_spatial", columns=cols, n_clusters=8
)
ward_spatial_clusters[["ward_spatial", "geometry"]].explore(
    "ward_spatial",
    categorical=True,
    style_kwds={"weight": 0.2},
    tiles="cartodb darkmatter",
    cmap="tab20b",
)

### Alternative Solution (SKATER)

In [ ]:
skater_clusters = gsp.analyze.regionalize(
    la20, method="skater", columns=cols, n_clusters=8
)
skater_clusters[["skater", "geometry"]].explore(
    "skater",
    categorical=True,
    style_kwds={"weight": 0.2},
    tiles="cartodb darkmatter",
    cmap="tab20b",
)